In [1]:
%load_ext autoreload
%autoreload 2

# VISTA — ResNet-34 Building Damage Assessment Pipeline

Imports directly from the original `.py` files — no code duplication.

**Required folder structure:**
```
xView2_first_place_3_classes/
├── zoo/
│   ├── __init__.py
│   ├── models.py
│   ├── senet.py
│   └── dpn.py
├── utils.py
├── losses.py
├── adamw.py
├── train34_loc.py
├── train34_cls.py
├── tune34_cls.py
├── predict34_loc.py
├── predict34cls.py
└── create_submission.py

visual-aid-le-wagon/xbd-dataset/xbd/
├── train/
│   ├── images/
│   └── masks/    ← generated by cell 2
└── test/
    └── images/
```

**Pipeline order:**
1. Install
2. Setup & config
3. Create masks
4. Train localization
5. Predict loc on val split *(required by train34_cls.ValData)*
6. Train classification
7. Fine-tune classification
8. Predict localization (test)
9. Predict classification (test)
10. Build submission
11. Visualize

## 1 · Install

In [2]:
import subprocess, sys

packages = [
    'opencv-python-headless', 'torch', 'torchvision',
    'scikit-image', 'scikit-learn', 'scipy',
    'pandas', 'tqdm', 'pretrainedmodels', 'matplotlib', 'shapely',
]
for pkg in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

# apex — required by train34_cls / tune34_cls for mixed-precision training
try:
    import apex
    print('apex already installed')
except ImportError:
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q',
        'git+https://github.com/NVIDIA/apex.git#egg=apex',
        '--no-build-isolation'
    ])

print('Done.')

apex already installed
Done.


## 2 · Setup & Config

In [3]:
import os, sys, random, gc
import numpy as np
import cv2
import torch
from torch import nn
from torch.utils.data import DataLoader
from torch.autograd import Variable
import torch.optim.lr_scheduler as lr_scheduler
from tqdm import tqdm
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from os import path, makedirs, listdir
from sklearn.model_selection import train_test_split

np.random.seed(1)
random.seed(1)
cv2.setNumThreads(0)
cv2.ocl.setUseOpenCL(False)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

Device: cuda


In [6]:
# ── Paths ─────────────────────────────────────────────────────────────────────
PROJECT_DIR  = 'xView2_first_place_3_classes'
TRAIN_IMGS   = 'visual-aid-le-wagon/xbd-dataset/xbd/train/images'
TRAIN_MASKS  = 'visual-aid-le-wagon/xbd-dataset/xbd/train/masks'
TRAIN_LABELS = 'visual-aid-le-wagon/xbd-dataset/xbd/train/labels'
TEST_DIR     = 'visual-aid-le-wagon/xbd-dataset/xbd/test/images'

WEIGHTS_DIR  = f'{PROJECT_DIR}/weights'
LOC_VAL_DIR  = f'{PROJECT_DIR}/pred_loc_val'   # loc preds on val split (needed by train34_cls)
PRED_LOC_DIR = f'{PROJECT_DIR}/pred34_loc'      # loc preds on test set
SUB_DIR      = f'{PROJECT_DIR}/submission'

# ── Hyperparameters ───────────────────────────────────────────────────────────
SEEDS          = [0]   # original uses [0,1,2]; start with [0] for baseline
LOC_EPOCHS     = 60
CLS_EPOCHS     = 30 #TODO change to 30 later
TUNE_EPOCHS    = 1 #TODO change to 3 later
BATCH_SIZE     = 4
VAL_BATCH_SIZE = 4

for d in [WEIGHTS_DIR, LOC_VAL_DIR, PRED_LOC_DIR, SUB_DIR]:
    makedirs(d, exist_ok=True)

# ── Python path ───────────────────────────────────────────────────────────────
sys.path.insert(0, PROJECT_DIR)
#sys.path.insert(0, f'{PROJECT_DIR}/zoo')

# ── Collect training files ────────────────────────────────────────────────────
all_files = sorted([
    path.join(TRAIN_IMGS, f)
    for f in listdir(TRAIN_IMGS)
    if '_pre_disaster.png' in f
])
print(f'Training images: {len(all_files)}')

Training images: 8494


In [7]:
# ── Imports from .py files ────────────────────────────────────────────────────
from utils import (
    shift_image, rotate_image, gauss_noise, clahe,
    saturation, brightness, contrast,
    change_hsv, shift_channels, AverageMeter, preprocess_inputs, dice
)
from losses import dice_round, ComboLoss
from adamw import AdamW
from zoo.models import Res34_Unet_Loc, Res34_Unet_Double

import train34_loc as t34_loc
import train34_cls as t34_cls
import tune34_cls  as tune34

print('All imports OK.')

All imports OK.


## 3 · Create Masks
Reads JSON labels → writes binary building masks + damage class masks into `train/masks/`.  
Run once. Skip if masks already exist.

In [8]:
import json
from multiprocessing import Pool
from shapely.wkt import loads as wkt_loads
comment = """
makedirs(TRAIN_MASKS, exist_ok=True)

damage_dict = {
    'no-damage': 1, 'minor-damage': 2,
    'major-damage': 3, 'destroyed': 4, 'un-classified': 1
}

label_files = sorted([
    path.join(TRAIN_LABELS, f.replace('_pre_disaster.png', '_pre_disaster.json'))
    for f in listdir(TRAIN_IMGS) if '_pre_disaster.png' in f
])

def process_image(json_file):
    js1 = json.load(open(json_file))
    js2 = json.load(open(json_file.replace('_pre_disaster', '_post_disaster')))
    msk     = np.zeros((1024, 1024), dtype='uint8')
    msk_dmg = np.zeros((1024, 1024), dtype='uint8')
    int_c   = lambda x: np.array(x).round().astype(np.int32)
    for feat in js1['features']['xy']:
        poly = wkt_loads(feat['wkt'])
        m = np.zeros((1024, 1024), np.uint8)
        cv2.fillPoly(m, [int_c(poly.exterior.coords)], 1)
        msk[m > 0] = 255
    for feat in js2['features']['xy']:
        poly    = wkt_loads(feat['wkt'])
        subtype = feat['properties']['subtype']
        m = np.zeros((1024, 1024), np.uint8)
        cv2.fillPoly(m, [int_c(poly.exterior.coords)], 1)
        msk_dmg[m > 0] = damage_dict[subtype]
    pre_out  = json_file.replace('/labels/', '/masks/').replace('_pre_disaster.json',  '_pre_disaster.png')
    post_out = json_file.replace('/labels/', '/masks/').replace('_pre_disaster.json', '_post_disaster.png')
    cv2.imwrite(pre_out,  msk,     [cv2.IMWRITE_PNG_COMPRESSION, 9])
    cv2.imwrite(post_out, msk_dmg, [cv2.IMWRITE_PNG_COMPRESSION, 9])

if label_files:
    with Pool() as pool:
        list(tqdm(pool.imap(process_image, label_files), total=len(label_files)))
    print(f'Masks created: {len(label_files)}')
else:
    print('No label files found — skipping.')"""

## 4 · Train Localization Model
Uses `train34_loc.TrainData`, `ValData`, `train_epoch`, `evaluate_val` directly.  
Saves → `weights/res34_loc_{seed}_1_best`

In [ ]:
# Patch module-level vars used inside train34_loc functions
t34_loc.all_files     = all_files
t34_loc.models_folder = WEIGHTS_DIR
t34_loc.input_shape   = (736, 736)

for seed in SEEDS:
    snap = f'res34_loc_{seed}_1'
    train_idxs, val_idxs = train_test_split(
        np.arange(len(all_files)), test_size=0.1, random_state=seed) 

    train_loader = DataLoader(
        t34_loc.TrainData(train_idxs),
        batch_size=BATCH_SIZE, num_workers=4,
        shuffle=True, pin_memory=False, drop_last=True)
    val_loader = DataLoader(
        t34_loc.ValData(val_idxs),
        batch_size=VAL_BATCH_SIZE, num_workers=4,
        shuffle=False, pin_memory=False)

    model     = Res34_Unet_Loc()
    optimizer = AdamW(model.parameters(), lr=0.00015, weight_decay=1e-6)
    scheduler = lr_scheduler.MultiStepLR(
        optimizer,
        milestones=[5,11,17,25,33,47,50,60,70,90,110,130,150,170,180,190],
        gamma=0.5)
    model    = nn.DataParallel(model).to(DEVICE)
    seg_loss = ComboLoss({'dice': 1.0, 'focal': 10.0}, per_image=False).to(DEVICE)

    best_score = 0
    for epoch in range(LOC_EPOCHS):
        t34_loc.train_epoch(epoch, seg_loss, model, optimizer, scheduler, train_loader)
        if epoch % 2 == 0:
            torch.cuda.empty_cache()
            best_score = t34_loc.evaluate_val(
                val_loader, best_score, model, snap, epoch)

    print(f'[seed={seed}] Loc done — best Dice: {best_score:.4f}')

/opt/conda/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet34_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet34_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
  0%|          | 0/1911 [00:00<?, ?it/s]/opt/conda/lib/python3.10/site-packages/torch/optim/lr_scheduler.py:746: UserWarning: To get the last learning rate computed by the scheduler, please use `get_last_lr()`.
  _warn_get_lr_called_within_step(self)
epoch: 0; lr 0.0001500; Loss 1.8123 (1.8123); Dice 0.0030 (0.0030):   0%|          | 0/1911 [00:01<?, ?it/s]/home/jupyter/xVie

epoch: 10; lr 0.0000750; Loss 0.3990; Dice 0.7209


100%|██████████| 213/213 [00:44<00:00,  4.81it/s]


Val Dice: 0.8410044618431299
score: 0.8410044618431299	score_best: 0.8410044618431299


epoch: 11; lr 0.0000750; Loss 0.2390 (0.3903); Dice 0.7900 (0.7235): 100%|██████████| 1911/1911 [10:45<00:00,  2.96it/s]


...

epoch: 58; lr 0.0000012; Loss 0.3428; Dice 0.7554


100%|██████████| 213/213 [00:44<00:00,  4.78it/s]


Val Dice: 0.8528504635207077
score: 0.8528504635207077	score_best: 0.8548231788865311


epoch: 59; lr 0.0000012; Loss 1.0022 (0.3446); Dice 0.0000 (0.7515): 100%|██████████| 1911/1911 [10:26<00:00,  3.05it/s]

epoch: 59; lr 0.0000012; Loss 0.3446; Dice 0.7515
[seed=0] Loc done — best Dice: 0.8548


## 5 · Predict Localization on Validation Split
`train34_cls.ValData` reads from `pred_loc_val/` to use as location priors during cls validation.  
This step generates those files. (Matches `predict_loc_val.py` logic, 2-flip TTA.)

In [9]:
def load_checkpoint(model, snap_path):
    ckpt = torch.load(snap_path, map_location='cpu', weights_only=False)
    sd   = model.state_dict()
    for k in sd:
        if k in ckpt['state_dict'] and sd[k].size() == ckpt['state_dict'][k].size():
            sd[k] = ckpt['state_dict'][k]
    model.load_state_dict(sd)
    print(f'  Loaded {path.basename(snap_path)} '
          f'(epoch {ckpt["epoch"]}, score {ckpt["best_score"]:.4f})')
    return model

makedirs(LOC_VAL_DIR, exist_ok=True)

for seed in SEEDS:
    _, val_idxs = train_test_split(
        np.arange(len(all_files)), test_size=0.1, random_state=seed)

    model = nn.DataParallel(Res34_Unet_Loc()).to(DEVICE)
    model = load_checkpoint(model, path.join(WEIGHTS_DIR, f'res34_loc_{seed}_1_best'))
    model.eval()

    print(f'[seed={seed}] Predicting loc on {len(val_idxs)} val images...')
    with torch.no_grad():
        for idx in tqdm(val_idxs):
            fn  = all_files[idx]
            img = preprocess_inputs(cv2.imread(fn, cv2.IMREAD_COLOR))
            inp = np.stack([img, img[::-1,::-1,...]], axis=0)
            inp = Variable(torch.from_numpy(
                inp.transpose(0,3,1,2)).float()).to(DEVICE)
            msk  = torch.sigmoid(model(inp)).cpu().numpy()
            pred = np.mean([msk[0,...], msk[1,:,::-1,::-1]], axis=0)
            out  = (pred * 255).astype('uint8').transpose(1,2,0)
            fname = fn.split('/')[-1].replace('.png', '_part1.png')
            cv2.imwrite(path.join(LOC_VAL_DIR, fname),
                        out[...,0], [cv2.IMWRITE_PNG_COMPRESSION, 9])

print('Val loc predictions saved to:', LOC_VAL_DIR)

/opt/conda/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet34_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet34_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


  Loaded res34_loc_0_1_best (epoch 41, score 0.8548)
[seed=0] Predicting loc on 850 val images...


100%|██████████| 850/850 [04:37<00:00,  3.07it/s]

Val loc predictions saved to: xView2_first_place_3_classes/pred_loc_val


## 6 · Train Classification Model
Uses `train34_cls.TrainData`, `ValData`, `train_epoch`, `evaluate_val` directly.  
Initialized from loc checkpoint.  

**Class balancing:** images with minor/major/destroyed are duplicated 2–3× in `train_idxs`.  
Loss coefficients per channel: `[0.05, 0.2, 0.8, 0.7, 0.4]`  
Saves → `weights/res34_cls2_{seed}_0_best`

In [ ]:
import logging
import csv

logging.basicConfig(
    filename=path.join(WEIGHTS_DIR, 'training.log'),
    level=logging.INFO,
    format='%(asctime)s %(message)s'
)

t34_cls.all_files     = all_files
t34_cls.models_folder = WEIGHTS_DIR
t34_cls.loc_folder    = LOC_VAL_DIR
t34_cls.input_shape   = (608, 608)

for seed in SEEDS:
    snap = f'res34_cls2_{seed}_0'
    print(f'[seed={seed}] Scanning class distribution...')
    file_classes = []
    for fn in tqdm(all_files):
        
        fl = np.zeros((3,), dtype=bool)
        msk = cv2.imread(
            fn.replace('/images/', '/masks/').replace('_pre_disaster', '_post_disaster'),
            cv2.IMREAD_UNCHANGED)

        fl[0] = (msk is not None) and (1 in msk)
        fl[1] = (msk is not None) and (2 in msk or 3 in msk)  # damaged
        fl[2] = (msk is not None) and (4 in msk)               # destroyed
        file_classes.append(fl)
    file_classes = np.asarray(file_classes)

    train_idxs0, val_idxs = train_test_split(
        np.arange(len(all_files)), test_size=0.1, random_state=seed)
    train_idxs = []
    for i in train_idxs0:
        train_idxs.append(i)
        if file_classes[i, 1:].max():   # any damaged or destroyed
            train_idxs.append(i)
        if file_classes[i, 1]:          # damaged specifically
            train_idxs.append(i)
    train_idxs = np.asarray(train_idxs)
    print(f'  train after balancing: {len(train_idxs)} (from {len(train_idxs0)})')

    train_loader = DataLoader(
        t34_cls.TrainData(train_idxs),
        batch_size=BATCH_SIZE, num_workers=4,
        shuffle=True, pin_memory=False, drop_last=True)
    val_loader = DataLoader(
        t34_cls.ValData(val_idxs),
        batch_size=VAL_BATCH_SIZE, num_workers=4,
        shuffle=False, pin_memory=False)

    model     = Res34_Unet_Double().to(DEVICE)
    optimizer = AdamW(model.parameters(), lr=0.0002, weight_decay=1e-6)
    model     = load_checkpoint(model, path.join(WEIGHTS_DIR, f'res34_loc_{seed}_1_best'))
    gc.collect(); torch.cuda.empty_cache()
    scheduler = lr_scheduler.MultiStepLR(
        optimizer,
        milestones=[5,11,17,23,29,33,47,50,60,70,90,110,130,150,170,180,190],
        gamma=0.5)
    model    = nn.DataParallel(model).to(DEVICE)
    seg_loss = ComboLoss({'dice': 1.0, 'focal': 12.0}, per_image=False).to(DEVICE)
    ce_loss  = nn.CrossEntropyLoss().to(DEVICE)

    # initialise csv log
    log_path = path.join(WEIGHTS_DIR, f'training_log_cls_seed{seed}.csv')
    with open(log_path, 'w', newline='') as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(['epoch', 'train_loss', 'train_dice', 'val_loss', 'val_score', 'dice', 
                 'f1', 'f1_0', 'f1_1', 'f1_2', 'precision', 'p_0', 'p_1', 'p_2'])

    best_score = 0
    for epoch in range(CLS_EPOCHS):
            train_loss, train_dice = t34_cls.train_epoch(epoch, seg_loss, ce_loss, model, optimizer, scheduler, train_loader)
            torch.cuda.empty_cache()
            best_score, metrics = t34_cls.evaluate_val(val_loader, best_score, model, snap, epoch, seg_loss)
            logging.info(f'[cls][seed={seed}] epoch={epoch} best_score={best_score:.4f}')
            with open(log_path, 'a', newline='') as csvfile:
                writer = csv.writer(csvfile)
                writer.writerow([
                        epoch,
                        f'{train_loss:.4f}',
                        f'{train_dice:.4f}',
                        f'{metrics["val_loss"]:.4f}',
                        f'{metrics["score"]:.4f}',
                        f'{metrics["dice"]:.4f}',
                        f'{metrics["f1"]:.4f}',
                        f'{metrics["f1_0"]:.4f}',
                        f'{metrics["f1_1"]:.4f}',
                        f'{metrics["f1_2"]:.4f}',
                        f'{metrics["precision"]:.4f}',
                        f'{metrics["p_0"]:.4f}',
                        f'{metrics["p_1"]:.4f}',
                        f'{metrics["p_2"]:.4f}',
                    ])

    print(f'[seed={seed}] Cls done — best precision: {best_score:.4f}')
    print(f'Log saved to: {log_path}')

[seed=0] Scanning class distribution...


100%|██████████| 8494/8494 [12:11<00:00, 11.62it/s]


  train after balancing: 12424 (from 7644)
  Loaded res34_loc_0_1_best (epoch 41, score 0.8548)


  0%|          | 0/3106 [00:00<?, ?it/s]/home/jupyter/xView2_first_place_3_classes/train34_cls.py:232: FutureWarning: `square` is deprecated since version 0.25 and will be removed in version 0.27. Use `skimage.morphology.footprint_rectangle` instead.
  msk[..., 1] = dilation(msk[..., 1], square(5))
/home/jupyter/xView2_first_place_3_classes/train34_cls.py:233: FutureWarning: `square` is deprecated since version 0.25 and will be removed in version 0.27. Use `skimage.morphology.footprint_rectangle` instead.
  msk[..., 2] = dilation(msk[..., 2], square(5))
/home/jupyter/xView2_first_place_3_classes/train34_cls.py:234: FutureWarning: `square` is deprecated since version 0.25 and will be removed in version 0.27. Use `skimage.morphology.footprint_rectangle` instead.
  msk[..., 3] = dilation(msk[..., 3], square(5))
/home/jupyter/xView2_first_place_3_classes/train34_cls.py:232: FutureWarning: `square` is deprecated since version 0.25 and will be removed in version 0.27. Use `skimage.morphology

epoch: 0; lr 0.0002000; Loss 1.5277; loss2 1.0367; Dice 0.5491


100%|██████████| 213/213 [02:27<00:00,  1.44it/s]


Val Score: 0.5860, Dice: 0.8508, Precision: 0.5722, F1: 0.4725, F1_0(no-dmg): 0.8603, F1_1(damaged): 0.2833, F1_2(destroyed): 0.6035, P_0: 0.8348, P_1: 0.3938, P_2: 0.4881
val_score: 0.5860	best_precision: 0.5722


  0%|          | 0/3106 [00:00<?, ?it/s]/home/jupyter/xView2_first_place_3_classes/train34_cls.py:232: FutureWarning: `square` is deprecated since version 0.25 and will be removed in version 0.27. Use `skimage.morphology.footprint_rectangle` instead.
  msk[..., 1] = dilation(msk[..., 1], square(5))
/home/jupyter/xView2_first_place_3_classes/train34_cls.py:233: FutureWarning: `square` is deprecated since version 0.25 and will be removed in version 0.27. Use `skimage.morphology.footprint_rectangle` instead.
  msk[..., 2] = dilation(msk[..., 2], square(5))
/home/jupyter/xView2_first_place_3_classes/train34_cls.py:234: FutureWarning: `square` is deprecated since version 0.25 and will be removed in version 0.27. Use `skimage.morphology.footprint_rectangle` instead.
  msk[..., 3] = dilation(msk[..., 3], square(5))
/home/jupyter/xView2_first_place_3_classes/train34_cls.py:232: FutureWarning: `square` is deprecated since version 0.25 and will be removed in version 0.27. Use `skimage.morphology

## 7 · Fine-tune Classification Model
Uses `tune34_cls.TrainData`, `ValData`, `train_epoch`, `evaluate_val` directly.  
Low LR, full data, reduced augmentation.  
Saves → `weights/res34_cls2_{seed}_tuned_best`

In [ ]:
"""import logging
import csv

logging.basicConfig(
    filename=path.join(WEIGHTS_DIR, 'training.log'),
    level=logging.INFO,
    format='%(asctime)s %(message)s'
)

tune34.all_files     = all_files
tune34.models_folder = WEIGHTS_DIR
tune34.loc_folder    = LOC_VAL_DIR
tune34.input_shape   = (608, 608)
tune34.train_len     = len(all_files)

for seed in SEEDS:
    snap = f'res34_cls2_{seed}_tuned'
    print(f'[seed={seed}] Scanning class distribution for fine-tune...')
    file_classes = []
    for fn in tqdm(all_files):
        fl = np.zeros((3,), dtype=bool)
        fl[0] = (msk is not None) and (1 in msk)
        fl[1] = (msk is not None) and (2 in msk or 3 in msk)  # damaged
        fl[2] = (msk is not None) and (4 in msk)               # destroyed
        msk = cv2.imread(
            fn.replace('/images/', '/masks/').replace('_pre_disaster', '_post_disaster'),
            cv2.IMREAD_UNCHANGED)
        file_classes.append(fl)
    file_classes = np.asarray(file_classes)

    train_idxs = []
    for i in np.arange(len(all_files)):
        train_idxs.append(i)
        if file_classes[i, 1:].max():   # any damaged or destroyed
            train_idxs.append(i)
        if file_classes[i, 1]:          # damaged specifically
            train_idxs.append(i)
    train_idxs = np.asarray(train_idxs)

    _, val_idxs = train_test_split(
        np.arange(len(all_files)), test_size=0.1, random_state=seed)
    print(f'  fine-tune samples: {len(train_idxs)}')

    train_loader = DataLoader(
        tune34.TrainData(train_idxs),
        batch_size=BATCH_SIZE, num_workers=4,
        shuffle=True, pin_memory=False, drop_last=True)
    val_loader = DataLoader(
        tune34.ValData(val_idxs),
        batch_size=VAL_BATCH_SIZE, num_workers=4,
        shuffle=False, pin_memory=False)

    model     = Res34_Unet_Double().to(DEVICE)
    optimizer = AdamW(model.parameters(), lr=0.000008, weight_decay=1e-6)
    model     = load_checkpoint(model, path.join(WEIGHTS_DIR, f'res34_cls2_{seed}_0_best'))
    gc.collect(); torch.cuda.empty_cache()
    scheduler = lr_scheduler.MultiStepLR(
        optimizer,
        milestones=[1,2,3,4,5,7,9,11,17,23,29,33,47,50,60,70,90,110,130,150,170,180,190],
        gamma=0.5)
    model    = nn.DataParallel(model).to(DEVICE)
    seg_loss = ComboLoss({'dice': 1.0, 'focal': 12.0}, per_image=False).to(DEVICE)
    ce_loss  = nn.CrossEntropyLoss().to(DEVICE)

    # initialise csv log
    log_path = path.join(WEIGHTS_DIR, f'training_log_tune_seed{seed}.csv')
    with open(log_path, 'w', newline='') as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(['epoch', 'val_score', 'dice', 'f1', 'f1_0', 'f1_1', 'f1_2', 'f1_3',
                         'precision', 'p_0', 'p_1', 'p_2'])

    best_score = 0
    for epoch in range(TUNE_EPOCHS):
        tune34.train_epoch(
            epoch, seg_loss, ce_loss, model, optimizer, scheduler, train_loader)
        torch.cuda.empty_cache()
        best_score, metrics = tune34.evaluate_val(
            val_loader, best_score, model, snap, epoch)
        logging.info(f'[tune][seed={seed}] epoch={epoch} best_score={best_score:.4f}')
        with open(log_path, 'a', newline='') as csvfile:
            writer = csv.writer(csvfile)
            writer.writerow([
                        epoch,
                        f'{metrics["score"]:.4f}',
                        f'{metrics["dice"]:.4f}',
                        f'{metrics["f1"]:.4f}',
                        f'{metrics["f1_0"]:.4f}',
                        f'{metrics["f1_1"]:.4f}',
                        f'{metrics["f1_2"]:.4f}',
                        f'{metrics["precision"]:.4f}',
                        f'{metrics["p_0"]:.4f}',
                        f'{metrics["p_1"]:.4f}',
                        f'{metrics["p_2"]:.4f}',
                    ])

    print(f'[seed={seed}] Fine-tune done — best precision: {best_score:.4f}')
    print(f'Log saved to: {log_path}') """

[seed=0] Scanning class distribution for fine-tune...


100%|██████████| 8494/8494 [08:03<00:00, 17.58it/s]
/opt/conda/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet34_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet34_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


  fine-tune samples: 11505
  Loaded res34_cls2_0_0_best (epoch 5, score 0.7352)


  0%|          | 0/2876 [00:00<?, ?it/s]/home/jupyter/xView2_first_place-master/tune34_cls.py:245: FutureWarning: `square` is deprecated since version 0.25 and will be removed in version 0.27. Use `skimage.morphology.footprint_rectangle` instead.
  msk[..., 1] = dilation(msk[..., 1], square(5))
/home/jupyter/xView2_first_place-master/tune34_cls.py:245: FutureWarning: `square` is deprecated since version 0.25 and will be removed in version 0.27. Use `skimage.morphology.footprint_rectangle` instead.
  msk[..., 1] = dilation(msk[..., 1], square(5))
/home/jupyter/xView2_first_place-master/tune34_cls.py:246: FutureWarning: `square` is deprecated since version 0.25 and will be removed in version 0.27. Use `skimage.morphology.footprint_rectangle` instead.
  msk[..., 2] = dilation(msk[..., 2], square(5))
/home/jupyter/xView2_first_place-master/tune34_cls.py:246: FutureWarning: `square` is deprecated since version 0.25 and will be removed in version 0.27. Use `skimage.morphology.footprint_recta

epoch: 0; lr 0.0000080; Loss 2.2658; loss2 1.0582; Dice 0.4421


...

  0%|          | 0/2876 [00:00<?, ?it/s]/home/jupyter/xView2_first_place-master/tune34_cls.py:245: FutureWarning: `square` is deprecated since version 0.25 and will be removed in version 0.27. Use `skimage.morphology.footprint_rectangle` instead.
  msk[..., 1] = dilation(msk[..., 1], square(5))
/home/jupyter/xView2_first_place-master/tune34_cls.py:246: FutureWarning: `square` is deprecated since version 0.25 and will be removed in version 0.27. Use `skimage.morphology.footprint_rectangle` instead.
  msk[..., 2] = dilation(msk[..., 2], square(5))
/home/jupyter/xView2_first_place-master/tune34_cls.py:247: FutureWarning: `square` is deprecated since version 0.25 and will be removed in version 0.27. Use `skimage.morphology.footprint_rectangle` instead.
  msk[..., 3] = dilation(msk[..., 3], square(5))
/home/jupyter/xView2_first_place-master/tune34_cls.py:248: FutureWarning: `square` is deprecated since version 0.25 and will be removed in version 0.27. Use `skimage.morphology.footprint_recta

epoch: 8; lr 0.0000001; Loss 1.9487; loss2 0.9737; Dice 0.6348


100%|██████████| 213/213 [02:12<00:00,  1.61it/s]


Val Score: 0.5789, Dice: 0.8508, Precision: 0.6609, F1: 0.4623, F1_0: 0.9157, F1_1: 0.2231, F1_2: 0.5550, F1_3: 0.7841, P_0: 0.8818, P_1: 0.4011, P_2: 0.5576, P_3: 0.8033
val_score: 0.5789	best_precision: 0.6780


  0%|          | 0/2876 [00:00<?, ?it/s]/home/jupyter/xView2_first_place-master/tune34_cls.py:245: FutureWarning: `square` is deprecated since version 0.25 and will be removed in version 0.27. Use `skimage.morphology.footprint_rectangle` instead.
  msk[..., 1] = dilation(msk[..., 1], square(5))
/home/jupyter/xView2_first_place-master/tune34_cls.py:246: FutureWarning: `square` is deprecated since version 0.25 and will be removed in version 0.27. Use `skimage.morphology.footprint_rectangle` instead.
  msk[..., 2] = dilation(msk[..., 2], square(5))
/home/jupyter/xView2_first_place-master/tune34_cls.py:247: FutureWarning: `square` is deprecated since version 0.25 and will be removed in version 0.27. Use `skimage.morphology.footprint_rectangle` instead.
  msk[..., 3] = dilation(msk[..., 3], square(5))
/home/jupyter/xView2_first_place-master/tune34_cls.py:248: FutureWarning: `square` is deprecated since version 0.25 and will be removed in version 0.27. Use `skimage.morphology.footprint_recta

## 8 · Predict Localization (test set)
4-flip TTA, ensemble over seeds.  
Saves → `pred34_loc/{filename}_part1.png`  
*(Matches `predict34_loc.py`)*

In [ ]:
loc_models = []
for seed in SEEDS:
    m = nn.DataParallel(Res34_Unet_Loc()).to(DEVICE)
    m = load_checkpoint(m, path.join(WEIGHTS_DIR, f'res34_loc_{seed}_1_best'))
    m.eval()
    loc_models.append(m)

test_files = sorted([f for f in listdir(TEST_DIR) if '_pre_' in f])
print(f'Test images: {len(test_files)}')

with torch.no_grad():
    for f in tqdm(test_files):
        img = preprocess_inputs(cv2.imread(path.join(TEST_DIR, f), cv2.IMREAD_COLOR))
        inp = np.stack([
            img, img[::-1,...], img[:,::-1,...], img[::-1,::-1,...]
        ], axis=0)
        inp = Variable(torch.from_numpy(
            inp.transpose(0,3,1,2)).float()).to(DEVICE)

        preds = []
        for m in loc_models:
            msk = torch.sigmoid(m(inp)).cpu().numpy()
            preds += [msk[0,...], msk[1,:,::-1,:],
                      msk[2,:,:,::-1], msk[3,:,::-1,::-1]]

        out = (np.mean(preds, axis=0) * 255).astype('uint8').transpose(1,2,0)
        cv2.imwrite(
            path.join(PRED_LOC_DIR, f.replace('.png', '_part1.png')),
            out[...,0], [cv2.IMWRITE_PNG_COMPRESSION, 9])

## 9 · Predict Classification (test set)
4-flip TTA per seed.  
Saves → `res34cls2_{seed}_tuned/{filename}_part1.png` + `_part2.png`  
*(Matches `predict34cls.py`)*

In [ ]:
with torch.no_grad():
    for seed in SEEDS:
        """pred_dir = f'{PROJECT_DIR}/res34cls2_{seed}_tuned'
        makedirs(pred_dir, exist_ok=True)

        m = nn.DataParallel(Res34_Unet_Double()).to(DEVICE)
        m = load_checkpoint(m, path.join(WEIGHTS_DIR, f'res34_cls2_{seed}_tuned_best'))"""

        pred_dir = f'{PROJECT_DIR}/res34cls2_{seed}_0'
        makedirs(pred_dir, exist_ok=True)
        m = nn.DataParallel(Res34_Unet_Double()).to(DEVICE)
        m = load_checkpoint(m, path.join(WEIGHTS_DIR, f'res34_cls2_{seed}_0_best'))
        m.eval()

        print(f'[seed={seed}] Predicting classification...')
        for f in tqdm(test_files):
            img  = cv2.imread(path.join(TEST_DIR, f), cv2.IMREAD_COLOR)
            img2 = cv2.imread(
                path.join(TEST_DIR, f.replace('_pre_', '_post_')), cv2.IMREAD_COLOR)
            img  = preprocess_inputs(np.concatenate([img, img2], axis=2))
            inp  = np.stack([
                img, img[::-1,...], img[:,::-1,...], img[::-1,::-1,...]
            ], axis=0)
            inp  = Variable(torch.from_numpy(
                inp.transpose(0,3,1,2)).float()).to(DEVICE)

            msk  = torch.sigmoid(m(inp)).cpu().numpy()
            pred = np.mean([
                msk[0,...], msk[1,:,::-1,:],
                msk[2,:,:,::-1], msk[3,:,::-1,::-1]
            ], axis=0)
            out  = (pred * 255).astype('uint8').transpose(1,2,0)
            cv2.imwrite(
                path.join(pred_dir, f.replace('.png', '_part1.png')),
                out[...,:3], [cv2.IMWRITE_PNG_COMPRESSION, 9])
            cv2.imwrite(
                path.join(pred_dir, f.replace('.png', '_part2.png')),
                out[...,2:], [cv2.IMWRITE_PNG_COMPRESSION, 9])

### Final validation metrics (best checkpoint)

In [ ]:
import csv
for seed in SEEDS:
    t34_cls.all_files     = all_files
    t34_cls.models_folder = WEIGHTS_DIR
    t34_cls.loc_folder    = LOC_VAL_DIR
    t34_cls.input_shape   = (608, 608)
    _, val_idxs = train_test_split(
        np.arange(len(all_files)), test_size=0.1, random_state=seed)
    val_loader = DataLoader(
        t34_cls.ValData(val_idxs),
        batch_size=VAL_BATCH_SIZE, num_workers=4,
        shuffle=False, pin_memory=False)
    m = nn.DataParallel(Res34_Unet_Double()).to(DEVICE)
    m = load_checkpoint(m, path.join(WEIGHTS_DIR, f'res34_cls2_{seed}_0_best'))
    m.eval()
    print(f'[seed={seed}]')
    _, metrics = t34_cls.evaluate_val(val_loader, -1, m, f'res34_cls2_{seed}_0', 0)
    log_path = path.join(WEIGHTS_DIR, f'final_metrics_cls_seed{seed}.csv')
    with open(log_path, 'w', newline='') as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(['val_score', 'dice', 'f1', 'f1_0', 'f1_1', 'f1_2',
                 'precision', 'p_0', 'p_1', 'p_2'])
        writer.writerow([
            f'{metrics["score"]:.4f}', f'{metrics["dice"]:.4f}',
            f'{metrics["f1"]:.4f}', f'{metrics["f1_0"]:.4f}',
            f'{metrics["f1_1"]:.4f}', f'{metrics["f1_2"]:.4f}',
            f'{metrics["precision"]:.4f}',
            f'{metrics["p_0"]:.4f}', f'{metrics["p_1"]:.4f}',
            f'{metrics["p_2"]:.4f}',
        ])
    print(f'Metrics saved to: {log_path}')

"""import csv

for seed in SEEDS:
    tune34.all_files     = all_files
    tune34.models_folder = WEIGHTS_DIR
    tune34.loc_folder    = LOC_VAL_DIR
    tune34.input_shape   = (608, 608)

    _, val_idxs = train_test_split(
        np.arange(len(all_files)), test_size=0.1, random_state=seed)
    val_loader = DataLoader(
        tune34.ValData(val_idxs),
        batch_size=VAL_BATCH_SIZE, num_workers=4,
        shuffle=False, pin_memory=False)
    m = nn.DataParallel(Res34_Unet_Double()).to(DEVICE)
    m = load_checkpoint(m, path.join(WEIGHTS_DIR, f'res34_cls2_{seed}_tuned_best'))
    m.eval()
    print(f'[seed={seed}]')
    _, metrics = tune34.evaluate_val(val_loader, -1, m, f'res34_cls2_{seed}_tuned', 0)

    log_path = path.join(WEIGHTS_DIR, f'final_metrics_tune_seed{seed}.csv')
    with open(log_path, 'w', newline='') as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(['val_score', 'dice', 'f1', 'f1_0', 'f1_1', 'f1_2', 'f1_3',
                         'precision', 'p_0', 'p_1', 'p_2', 'p_3'])
        writer.writerow([
            f'{metrics["score"]:.4f}',
            f'{metrics["dice"]:.4f}',
            f'{metrics["f1"]:.4f}',
            f'{metrics["f1_0"]:.4f}',
            f'{metrics["f1_1"]:.4f}',
            f'{metrics["f1_2"]:.4f}',
            f'{metrics["f1_3"]:.4f}',
            f'{metrics["precision"]:.4f}',
            f'{metrics["p_0"]:.4f}',
            f'{metrics["p_1"]:.4f}',
            f'{metrics["p_2"]:.4f}',
            f'{metrics["p_3"]:.4f}',
        ])
    print(f'Metrics saved to: {log_path}')"""

## 10 · Build Submission
Merges loc + cls with thresholds from `create_submission.py`.  
Saves → `submission/{filename}_localization_prediction.png` + `_damage_prediction.png`

In [ ]:
import shutil
shutil.rmtree(SUB_DIR)
makedirs(SUB_DIR, exist_ok=True)
print('SUB_DIR cleared')

In [ ]:
from skimage.morphology import square, dilation

_thr         = [0.38, 0.13, 0.14]
#pred_folders = [f'{PROJECT_DIR}/res34cls2_{s}_tuned' for s in SEEDS]
pred_folders = [f'{PROJECT_DIR}/res34cls2_{s}_0' for s in SEEDS]
loc_folders  = [PRED_LOC_DIR]
makedirs(SUB_DIR, exist_ok=True)

def process_submission(f):
    # Classification ensemble
    preds = []
    for d in pred_folders:
        msk1 = cv2.imread(path.join(d, f), cv2.IMREAD_UNCHANGED)
        msk2 = cv2.imread(path.join(d, f.replace('_part1', '_part2')), cv2.IMREAD_UNCHANGED)
        preds.append(np.concatenate([msk1, msk2[...,1:]], axis=2))
    preds = np.mean(preds, axis=0).astype('float') / 255

    # Localization ensemble
    loc_preds = np.mean([
        cv2.imread(path.join(d, f), cv2.IMREAD_UNCHANGED)
        for d in loc_folders
    ], axis=0).astype('float') / 255

    msk_dmg = preds[...,1:].argmax(axis=2) + 1
    msk_loc = (1 * (
        (loc_preds > _thr[0]) |
        ((loc_preds > _thr[1]) & (msk_dmg > 1) & (msk_dmg < 3)) |
        ((loc_preds > _thr[2]) & (msk_dmg > 1))
    )).astype('uint8')
    msk_dmg = (msk_dmg * msk_loc).astype('uint8')

    _minor = (msk_dmg == 2)
    if _minor.sum() > 0:
        msk_dmg[dilation(_minor, square(5)) & (msk_dmg == 1)] = 2

    loc_name = f.replace('_pre_disaster_part1.png', '_localization_prediction.png')
    dmg_name = f.replace('_pre_disaster_part1.png', '_damage_prediction.png')
    cv2.imwrite(path.join(SUB_DIR, loc_name), msk_loc, [cv2.IMWRITE_PNG_COMPRESSION, 9])
    cv2.imwrite(path.join(SUB_DIR, dmg_name), msk_dmg, [cv2.IMWRITE_PNG_COMPRESSION, 9])

sub_files = sorted([f for f in listdir(pred_folders[0]) if '_part1.png' in f])
for f in tqdm(sub_files):
    process_submission(f)

print(f'Submission saved to: {SUB_DIR}  ({len(sub_files)} images)')

## 11 · Visualize

In [ ]:
DAMAGE_LABELS = {0:'background', 1:'no-damage', 2:'damaged', 3:'destroyed'}
DAMAGE_COLORS = np.array(
    [[0,0,0],[0,255,0],[255,128,0],[255,0,0]], dtype='uint8')

def visualize(pre_path):
    f_base = path.basename(pre_path)
    pre    = cv2.cvtColor(cv2.imread(pre_path), cv2.COLOR_BGR2RGB)
    post   = cv2.cvtColor(
        cv2.imread(pre_path.replace('_pre_', '_post_')), cv2.COLOR_BGR2RGB)
    loc_p  = path.join(SUB_DIR,
        f_base.replace('_pre_disaster.png', '_localization_prediction.png'))
    dmg_p  = path.join(SUB_DIR,
        f_base.replace('_pre_disaster.png', '_damage_prediction.png'))

    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    axes[0].imshow(pre);  axes[0].set_title('Pre');  axes[0].axis('off')
    axes[1].imshow(post); axes[1].set_title('Post'); axes[1].axis('off')

    if path.exists(loc_p):
        axes[2].imshow(cv2.imread(loc_p, cv2.IMREAD_UNCHANGED), cmap='gray')
    axes[2].set_title('Localization'); axes[2].axis('off')

    if path.exists(dmg_p):
        dmg = cv2.imread(dmg_p, cv2.IMREAD_UNCHANGED)
        axes[3].imshow(DAMAGE_COLORS[np.clip(dmg, 0, 4)])
        patches = [
            mpatches.Patch(color=DAMAGE_COLORS[k]/255, label=DAMAGE_LABELS[k])
            for k in range(1, 4)
        ]
        axes[3].legend(handles=patches, loc='lower right', fontsize=8)
    axes[3].set_title('Damage'); axes[3].axis('off')

    plt.tight_layout()
    plt.show()

if test_files:
    visualize(path.join(TEST_DIR, test_files[542]))